# Stage 5 — QLoRA fine-tuning of SmolLM3-3B

Fine-tunes `HuggingFaceTB/SmolLM3-3B` with QLoRA on one of two training sets (decision D16):

- `TRAINING_SET = "stars"`: 15,000 B2W pool reviews (5,000 per class), star labels, 1 epoch;
- `TRAINING_SET = "human"`: 1,500 RePro dev reviews, human labels, 3 epochs.

**Run the notebook once per training set**: set `TRAINING_SET` in the configuration cell.
Checkpoints are compared on the other 500 dev reviews (the *selection* set), where the earlier
models are re-scored from their saved predictions. Decisions: D15–D16 in `docs/decisions.md`.

Runs on **Google Colab** or **Kaggle**; the first code cell detects which one.

- **Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
- **Kaggle:** in *Settings*, set *Accelerator → GPU T4 x2* and turn *Internet* on, then
  *Save Version → Save & Run All*. The zips appear in the version's *Output* tab.

What this notebook does:

1. Clones the repository, installs `peft` and `bitsandbytes`, rebuilds and checks the data.
2. Loads the model in 4 bits (NF4) and adds LoRA adapters (r 16) to every linear layer.
3. **Pilot:** times a few training steps and **stops** if the full run would take more than
   `MAX_HOURS` (nothing is reduced automatically).
4. Trains with the loss only on the answer tokens, evaluating on the selection set at 0%, 25%,
   50%, 75% and 100% of training; keeps the best trained checkpoint.
5. Compares it with TF-IDF, BERTimbau and the zero-shot base model on the same 500 reviews, and
   predicts the **test** set without test metrics (decision D12).
6. Saves `qlora_<set>_results.zip` (report and predictions) and `qlora_<set>_adapter.zip`
   (LoRA weights, for publishing in Stage 8; not committed to git).

In [ ]:
import importlib.util
import os
import subprocess
import sys

import torch

# Where are we? Kaggle sets KAGGLE_KERNEL_RUN_TYPE; Colab ships the google.colab package.
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    PLATFORM = "kaggle"
    WORK_DIR = "/tmp"                # scratch space: the clone and data are not kept as output
    OUTPUT_DIR = "/kaggle/working"   # only files here are saved with the notebook version
elif importlib.util.find_spec("google.colab") is not None:
    PLATFORM = "colab"
    WORK_DIR = OUTPUT_DIR = "/content"
else:
    raise RuntimeError("Run this notebook on Google Colab or Kaggle")
print("platform:", PLATFORM)

assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2"
)
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME)

REPO_URL = "https://github.com/PedroRicci12/ptbr-review-lora.git"
REPO_DIR = f"{WORK_DIR}/ptbr-review-lora"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository commit:", REPO_COMMIT)

# peft (LoRA) and bitsandbytes (4-bit weights) are not always preinstalled; versions go to the report.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes"], check=True)

## 1. Data

Same script as on the laptop. If the counts differ from the committed
`reports/data_summary.json`, the cell stops.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/prepare_data.py"],
    check=True, stdout=subprocess.DEVNULL, env={**os.environ, "PYTHONPATH": "src"},
)
diff = subprocess.run(
    ["git", "diff", "--exit-code", "reports/data_summary.json"], capture_output=True, text=True
)
assert diff.returncode == 0, "Prepared data differs from the repository:\n" + diff.stdout
print("data_summary.json identical to the repository")

## 2. Configuration

**Set `TRAINING_SET` before running.** Everything else is fixed (D16), not tuned.

In [ ]:
TRAINING_SET = "stars"   # "stars" (A) or "human" (B)
assert TRAINING_SET in ("stars", "human")

MODEL_ID = "HuggingFaceTB/SmolLM3-3B"
MODEL_REVISION = "a07cc9a04f16550a088caea529712d1d335b0ac1"  # same as Stage 4

SEED = 42
USE_TITLE = True
STARS_PER_CLASS = 5000                    # A: 15,000 reviews
EPOCHS = {"stars": 1, "human": 3}[TRAINING_SET]
DEV_TRAIN_SIZE = 1500                     # B: the rest of dev (500) is the selection set
MAX_TRAIN_TOKENS = 512                    # longer training examples are dropped (counted in the report)

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
MICRO_BATCH = 4
GRAD_ACCUM = 4                            # effective batch = 16
MAX_GRAD_NORM = 1.0
CHECKPOINTS = (0.25, 0.5, 0.75, 1.0)      # fractions of training evaluated on the selection set

PILOT_STEPS = 10                          # optimizer steps timed before committing to the full run
MAX_HOURS = 3.0
SCORE_BATCH = 8
N_RESAMPLES = 1000
REFERENCES = ("tfidf_lr__title_text__balanced", "bertimbau", "smollm3_base__zero_shot")
NAME = f"smollm3_qlora_{TRAINING_SET}"

import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from pathlib import Path

import pandas as pd

from ptbr_review_lora.data import LABELS, model_input, split_dev
from ptbr_review_lora.training import sample_balanced

pool = pd.read_parquet("data/processed/b2w_pool.parquet")
dev = pd.read_parquet("data/processed/repro_dev.parquet")
test = pd.read_parquet("data/processed/repro_test.parquet")

dev_train, select = split_dev(dev, train_size=DEV_TRAIN_SIZE, seed=SEED)
train_df = sample_balanced(pool, STARS_PER_CLASS, seed=SEED) if TRAINING_SET == "stars" else dev_train.sample(frac=1, random_state=SEED)

train_texts = model_input(train_df, USE_TITLE).tolist()
train_labels = train_df["label"].tolist()
select_texts = model_input(select, USE_TITLE).tolist()
y_select = select["label"].tolist()
SELECT_ROWS = select.index.tolist()   # positions in dev = `row` of the dev prediction files
test_texts = model_input(test, USE_TITLE).tolist()

print(TRAINING_SET, "training rows:", len(train_df), train_df["label"].value_counts().to_dict())
print("selection rows:", len(select), select["polarity_group"].value_counts().to_dict())

## 3. Model in 4 bits + LoRA adapters

- **NF4 + double quantization:** the frozen weights are stored in 4 bits (~2 GB instead of ~6 GB)
  and dequantized on the fly; computation runs in fp16 (the T4 has no bf16).
- **LoRA on every linear layer** (`all-linear`, the output layer excluded): each frozen matrix
  `W` gets a trainable `B·A` of rank 16, scaled by `alpha / r = 2`. `B` starts at zero, so at
  step 0 the model is exactly the (quantized) base model.
- `prepare_model_for_kbit_training` turns on gradient checkpointing (recomputes activations in
  the backward pass instead of storing them: less memory, more time).

In [ ]:
import peft
import transformers
from packaging.version import Version
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, prepare_model_for_kbit_training, set_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import bitsandbytes

assert Version(transformers.__version__) >= Version("4.53"), "SmolLM3 needs transformers >= 4.53"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
assert END_OF_TURN_ID != tokenizer.unk_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, quantization_config=bnb_config, dtype=torch.float16, device_map={"": 0}
)
model.config.use_cache = False  # the cache is for generation; it conflicts with gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules="all-linear", bias="none", task_type="CAUSAL_LM",
))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {trainable:,} of {total:,} ({trainable / total:.2%})")

# The LoRA layers live inside the base model, so these modules already include the adapters.
base = model.get_base_model()
decoder = base.get_decoder()            # transformer layers + final norm
lm_head = base.get_output_embeddings()  # hidden state -> vocabulary logits

In [ ]:
from ptbr_review_lora.prompts import build_messages
from ptbr_review_lora.training import build_example, pad_batch


def render(review):
    """Zero-shot prompt of Stage 4 (no date, thinking off), ending where the answer starts."""
    messages = build_messages(review)
    messages[0]["content"] += "\n/system_override"
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)


def encode_prompts(texts):
    return [tokenizer(render(t), add_special_tokens=False)["input_ids"] for t in texts]


LABEL_IDS = [tokenizer(label, add_special_tokens=False)["input_ids"] for label in LABELS]
ANSWER_IDS = {label: ids + [END_OF_TURN_ID] for label, ids in zip(LABELS, LABEL_IDS)}  # the model learns to stop

examples = [build_example(p, ANSWER_IDS[y]) for p, y in zip(encode_prompts(train_texts), train_labels)]
n_long = sum(len(e["input_ids"]) > MAX_TRAIN_TOKENS for e in examples)
examples = [e for e in examples if len(e["input_ids"]) <= MAX_TRAIN_TOKENS]
print(f"training examples: {len(examples)} (dropped {n_long} longer than {MAX_TRAIN_TOKENS} tokens)")

STEPS_PER_EPOCH = len(examples) // (MICRO_BATCH * GRAD_ACCUM)
TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
EVAL_AT = sorted({max(1, round(f * TOTAL_STEPS)) for f in CHECKPOINTS})
print("optimizer steps:", TOTAL_STEPS, "evaluations at:", EVAL_AT)

## 4. Label scoring on the selection set

Same method as Stage 4: for each review, the log-probability of each label as the answer; the
highest wins. The first check compares the shortcut with plain `model(...).logits`.

In [ ]:
import time

from ptbr_review_lora.metrics import bootstrap_ci, compute_metrics, macro_f1, neutral_recall_by_group
from ptbr_review_lora.prompts import choose_label


@torch.inference_mode()
def score_batch(prompts):
    seqs = [prompt + label_ids for prompt in prompts for label_ids in LABEL_IDS]
    input_ids = torch.full((len(seqs), max(map(len, seqs))), PAD_ID)
    attention_mask = torch.zeros_like(input_ids)
    rows, positions, targets = [], [], []
    for s, seq in enumerate(seqs):
        input_ids[s, : len(seq)] = torch.tensor(seq)
        attention_mask[s, : len(seq)] = 1
        prompt_len = len(prompts[s // len(LABELS)])
        for k, token in enumerate(LABEL_IDS[s % len(LABELS)]):
            rows.append(s)
            positions.append(prompt_len - 1 + k)
            targets.append(token)
    with torch.autocast("cuda", dtype=torch.float16):
        hidden = decoder(input_ids=input_ids.cuda(), attention_mask=attention_mask.cuda()).last_hidden_state
        rows_t = torch.tensor(rows, device="cuda")
        logits = lm_head(hidden[rows_t, torch.tensor(positions, device="cuda")])
    log_probs = torch.log_softmax(logits.float(), dim=-1)
    token_scores = log_probs[torch.arange(len(rows), device="cuda"), torch.tensor(targets, device="cuda")]
    scores = torch.zeros(len(seqs), device="cuda").index_add_(0, rows_t, token_scores)
    return scores.view(len(prompts), len(LABELS)).cpu().numpy()


@torch.inference_mode()
def score_plain(prompt):
    out = []
    for label_ids in LABEL_IDS:
        ids = torch.tensor([prompt + label_ids], device="cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=ids).logits[0]
        log_probs = torch.log_softmax(logits.float(), dim=-1)
        out.append(sum(log_probs[len(prompt) - 1 + k, t].item() for k, t in enumerate(label_ids)))
    return np.array(out)


def predict(prompts):
    was_training = model.training
    model.eval()  # no dropout while scoring
    order = np.argsort([-len(p) for p in prompts])
    scores = np.empty((len(prompts), len(LABELS)))
    for start in range(0, len(prompts), SCORE_BATCH):
        idx = order[start : start + SCORE_BATCH]
        scores[idx] = score_batch([prompts[i] for i in idx])
    model.train(was_training)
    assert np.isfinite(scores).all(), "non-finite scores"
    return [choose_label(dict(zip(LABELS, row))) for row in scores]


select_prompts = encode_prompts(select_texts)
model.eval()
fast, plain = score_batch(select_prompts[:4]), np.stack([score_plain(p) for p in select_prompts[:4]])
print("max |shortcut - plain|:", float(np.abs(fast - plain).max()))
assert np.allclose(fast, plain, atol=0.05) and (fast.argmax(1) == plain.argmax(1)).all()

## 5. Training

- Plain PyTorch loop (as for BERTimbau): fp16 autocast with a gradient scaler, AdamW on the
  LoRA weights only, linear warmup and decay, gradient clipping.
- The loss is cross-entropy on the **answer tokens only** (label + end of turn): the output layer
  runs only on the positions that predict them, as in the scoring.
- Step 0 is the base model in 4 bits: comparing it with the fp16 zero-shot of Stage 4 shows what
  quantization alone costs.
- After the pilot steps the cell **stops** if the estimated total time exceeds `MAX_HOURS`.

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=0.0)
scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(WARMUP_RATIO * TOTAL_STEPS)), TOTAL_STEPS)
scaler = torch.amp.GradScaler("cuda")


def answer_loss(batch):
    """Mean cross-entropy over the answer tokens of a padded batch."""
    input_ids = torch.tensor(batch["input_ids"], device="cuda")
    attention_mask = torch.tensor(batch["attention_mask"], device="cuda")
    labels = torch.tensor(batch["labels"], device="cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        hidden = decoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        targets = labels[:, 1:]          # the token at position i+1 is predicted from position i
        keep = targets != -100
        logits = lm_head(hidden[:, :-1][keep])
    return torch.nn.functional.cross_entropy(logits.float(), targets[keep])


def evaluate_select(step):
    preds = predict(select_prompts)
    return preds, {"step": step, "select_macro_f1": macro_f1(y_select, preds)}


history, checkpoints = [], {}
preds0, record = evaluate_select(0)
history.append({**record, "train_loss": None, "elapsed_seconds": 0})
print(history[-1])

rng = np.random.default_rng(SEED)
model.train()
step, start, losses, pilot = 0, time.perf_counter(), [], {}
for epoch in range(EPOCHS):
    order = rng.permutation(len(examples))
    micro_batches = [order[i : i + MICRO_BATCH] for i in range(0, STEPS_PER_EPOCH * MICRO_BATCH * GRAD_ACCUM, MICRO_BATCH)]
    for m, idx in enumerate(micro_batches):
        loss = answer_loss(pad_batch([examples[i] for i in idx], PAD_ID)) / GRAD_ACCUM
        scaler.scale(loss).backward()
        losses.append(loss.item() * GRAD_ACCUM)
        if (m + 1) % GRAD_ACCUM:
            continue
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step == PILOT_STEPS:
            per_step = (time.perf_counter() - start) / PILOT_STEPS
            estimate_hours = per_step * TOTAL_STEPS / 3600
            pilot = {"seconds_per_step": round(per_step, 2), "estimated_training_hours": round(estimate_hours, 2)}
            print("pilot:", pilot)
            assert estimate_hours <= MAX_HOURS, (
                f"Training would take ~{estimate_hours:.1f} h (limit {MAX_HOURS} h): reduce the training set and rerun"
            )
        if step in EVAL_AT:
            preds, record = evaluate_select(step)
            history.append({**record, "train_loss": float(np.mean(losses)), "elapsed_seconds": round(time.perf_counter() - start)})
            losses = []
            checkpoints[step] = {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}
            print(history[-1])

TRAIN_SECONDS = round(time.perf_counter() - start)
BEST_STEP = max(checkpoints, key=lambda s: next(h["select_macro_f1"] for h in history if h["step"] == s))
set_peft_model_state_dict(model, checkpoints[BEST_STEP])
print(f"best trained checkpoint: step {BEST_STEP} of {TOTAL_STEPS}; training time {TRAIN_SECONDS / 60:.1f} min")

## 6. Evaluation on the selection set and comparison

All models on the same 500 reviews: the earlier ones re-scored from their saved dev predictions
(`row` = position in dev), compared with the paired bootstrap (D11). With 500 reviews the
intervals are about twice as wide as on the full dev.

In [ ]:
from ptbr_review_lora.metrics import paired_bootstrap
from ptbr_review_lora.predictions import load_predictions, save_predictions


def evaluate(preds):
    _, low, high = bootstrap_ci(y_select, preds, n_resamples=N_RESAMPLES, seed=SEED)
    return {
        **compute_metrics(y_select, preds),
        "macro_f1_ci95": [low, high],
        "neutral_recall_by_group": neutral_recall_by_group(select["polarity_group"], preds),
    }


best_preds = predict(select_prompts)
results = {NAME: evaluate(best_preds), "step_0_base_4bit": evaluate(preds0)}
comparisons = {}
for ref in REFERENCES:
    ref_all = load_predictions(Path(f"predictions/dev/{ref}.csv"), len(dev))
    ref_preds = [ref_all[r] for r in SELECT_ROWS]
    results[ref] = evaluate(ref_preds)
    comparisons[f"{NAME} vs {ref}"] = paired_bootstrap(y_select, ref_preds, best_preds, n_resamples=N_RESAMPLES, seed=SEED)

for name, r in results.items():
    print(f"{name}: macro-F1 {r['macro_f1']:.4f} (95% CI {r['macro_f1_ci95'][0]:.4f}-{r['macro_f1_ci95'][1]:.4f})")
for name, c in comparisons.items():
    print(f"{name}: diff {c['diff']:+.4f} (95% CI {c['low']:+.4f} to {c['high']:+.4f}), p_b_not_better {c['p_b_not_better']:.3f}")

## 7. Test predictions with the best checkpoint (no metrics)

Decision D12: the test set is scored once, in Stage 6, for all frozen models together.

In [ ]:
start = time.perf_counter()
test_preds = predict(encode_prompts(test_texts))
TEST_SECONDS = round(time.perf_counter() - start)

# Selection-set predictions: `row` 0..499 follows SELECT_ROWS (sorted dev positions).
save_predictions(Path(f"predictions/dev_select/{NAME}.csv"), best_preds)
save_predictions(Path(f"predictions/test/{NAME}.csv"), test_preds)
print(len(test_preds), "test predictions,", TEST_SECONDS, "s")

## 8. Report, adapter and download

In [ ]:
import json
import platform
import shutil
import zipfile

import sklearn

report = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "repo_commit": REPO_COMMIT,
    "training_set": TRAINING_SET,
    "environment": {
        "platform": PLATFORM, "gpu": GPU_NAME, "python": platform.python_version(),
        "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
        "bitsandbytes": bitsandbytes.__version__, "scikit_learn": sklearn.__version__, "pandas": pd.__version__,
    },
    "config": {
        "epochs": EPOCHS, "stars_per_class": STARS_PER_CLASS if TRAINING_SET == "stars" else None,
        "dev_train_size": DEV_TRAIN_SIZE, "max_train_tokens": MAX_TRAIN_TOKENS, "use_title": USE_TITLE,
        "quantization": "nf4 + double quant, fp16 compute", "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT, "target_modules": "all-linear", "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
        "max_grad_norm": MAX_GRAD_NORM, "seed": SEED,
    },
    "trainable_parameters": trainable,
    "training_examples": len(examples),
    "training_label_counts": pd.Series(train_labels).value_counts().to_dict(),
    "dropped_long_examples": n_long,
    "total_steps": TOTAL_STEPS,
    "pilot": pilot,
    "history": history,
    "best_step": BEST_STEP,
    "train_seconds": TRAIN_SECONDS,
    "test_seconds": TEST_SECONDS,
    "split": "repro_dev_select",
    "select_rows": SELECT_ROWS,
    "n": len(select),
    "n_resamples": N_RESAMPLES,
    "results": results,
    "paired_comparisons": comparisons,
}
Path(f"reports/{NAME}_select.json").write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

RESULTS_ZIP = f"{OUTPUT_DIR}/qlora_{TRAINING_SET}_results.zip"
with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in (f"reports/{NAME}_select.json", f"predictions/dev_select/{NAME}.csv", f"predictions/test/{NAME}.csv"):
        zf.write(path)  # relative paths, so the files unpack into the same folders of the repository

ADAPTER_DIR = f"{WORK_DIR}/{NAME}_adapter"
model.save_pretrained(ADAPTER_DIR)  # LoRA weights + adapter config only (the base model is not copied)
tokenizer.save_pretrained(ADAPTER_DIR)
ADAPTER_ZIP = shutil.make_archive(f"{OUTPUT_DIR}/qlora_{TRAINING_SET}_adapter", "zip", ADAPTER_DIR)
print("saved", RESULTS_ZIP, "and", ADAPTER_ZIP)

if PLATFORM == "colab":
    from google.colab import files

    files.download(RESULTS_ZIP)
    files.download(ADAPTER_ZIP)
else:
    print("Kaggle: download both zips from the Output tab of this notebook version")

## Next step

Hand `qlora_<set>_results.zip` over: its files go into the repository at the same paths.
Keep `qlora_<set>_adapter.zip` outside the repository; it is published on Hugging Face in Stage 8.
Then run this notebook again with the other `TRAINING_SET`.